# Exemplo 2 — CRAG Básico (sem LangGraph)

**Aula 8 · MBA em RAG & CAG Aplicados a Direito e Segurança Pública**

---

## Objetivo

Demonstrar o conceito central do CRAG de forma linear e direta — sem usar LangGraph. O foco é entender o **mecanismo de avaliação de relevância + roteamento** antes de orquestrar com grafos.

## O que você vai aprender

- Como implementar um avaliador de relevância simples (LLM-as-Judge)
- Como rotear para web search quando o score é baixo
- Como fundir resultados locais e web antes de gerar

> **Próximo passo:** No LAB2, este mesmo pipeline será refatorado para usar LangGraph com StateGraph, nós e arestas condicionais.

## Passo 1 — Instalação

In [ ]:
!pip install langchain langchain-community langchain-openai chromadb sentence-transformers tavily-python -q
print("Instalação concluída!")

In [ ]:
import os
from google.colab import userdata

# Configurar chaves de API
# No Colab: Secrets → adicionar OPENAI_API_KEY e TAVILY_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
print("Chaves configuradas!")

## Passo 2 — Construir o Índice Vetorial

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

# Corpus jurídico para retrieval local
corpus = [
    Document(
        page_content="Art. 206, §3º, V do Código Civil: prescreve em 3 anos a pretensão de reparação civil. Este prazo aplica-se às ações de indenização por danos morais e materiais.",
        metadata={"fonte": "Código Civil", "artigo": "206"}
    ),
    Document(
        page_content="Art. 186 do Código Civil: aquele que por ação ou omissão voluntária, negligência ou imprudência, violar direito e causar dano a outrem, ainda que exclusivamente moral, comete ato ilícito.",
        metadata={"fonte": "Código Civil", "artigo": "186"}
    ),
    Document(
        page_content="Súmula 227 STJ: A pessoa jurídica pode sofrer dano moral.",
        metadata={"fonte": "STJ", "tipo": "súmula"}
    ),
    Document(
        page_content="Lei Maria da Penha (Lei 11.340/2006), Art. 5º: configura violência doméstica qualquer ação baseada em gênero que cause morte, lesão, sofrimento físico, sexual ou psicológico e dano moral ou patrimonial.",
        metadata={"fonte": "Lei Maria da Penha", "lei": "11.340/2006"}
    ),
]

# Embeddings e índice vetorial
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = Chroma.from_documents(corpus, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Índice pronto com {len(corpus)} documentos.")

## Passo 3 — Avaliador de Relevância (LLM-as-Judge)

O coração do CRAG: um LLM que avalia a qualidade dos documentos recuperados.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Prompt do avaliador
PROMPT_AVALIADOR = ChatPromptTemplate.from_template("""
Você é um avaliador de relevância para sistemas de busca jurídica.

Avalie se o documento abaixo é relevante para responder à pergunta.

Pergunta: {question}
Documento: {document}

Retorne APENAS um JSON válido com:
- "score": número entre 0.0 e 1.0 (1.0 = altamente relevante, 0.0 = irrelevante)
- "justificativa": explicação em uma frase

Exemplo: {{"score": 0.8, "justificativa": "O documento contém o artigo específico sobre o prazo solicitado."}}

JSON:
""")

def avaliar_relevancia(pergunta: str, documento: str) -> dict:
    """Avalia a relevância de um documento para uma pergunta. Retorna score 0-1."""
    prompt = PROMPT_AVALIADOR.format_messages(
        question=pergunta,
        document=documento[:500]  # Limitar para evitar tokens excessivos
    )
    resposta = llm.invoke(prompt)
    
    try:
        # Tentar parsear o JSON da resposta
        texto = resposta.content.strip()
        # Limpar possíveis marcadores markdown
        if texto.startswith("```"):
            texto = texto.split("```")[1]
            if texto.startswith("json"):
                texto = texto[4:]
        resultado = json.loads(texto)
        return resultado
    except:
        return {"score": 0.5, "justificativa": "Não foi possível parsear a avaliação"}


# Testar o avaliador
print("Testando avaliador de relevância...")
teste = avaliar_relevancia(
    pergunta="Qual o prazo prescricional para danos morais?",
    documento="Art. 206, §3º, V do CC: prescreve em 3 anos a pretensão de reparação civil."
)
print(f"Score: {teste['score']} | Justificativa: {teste['justificativa']}")

## Passo 4 — Web Search com Tavily

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Configurar Tavily
tavily = TavilySearchResults(
    max_results=3,
    search_depth="advanced"
)

def buscar_web(pergunta: str) -> list:
    """Realiza busca web com Tavily. Retorna lista de resultados."""
    resultados = tavily.invoke({"query": pergunta + " direito brasileiro"})
    return resultados

print("Tavily configurado!")

# Testar busca web
print("\nTestando busca web...")
resultados_web = buscar_web("novas decisões STF responsabilidade civil 2024")
print(f"Resultados obtidos: {len(resultados_web)}")
if resultados_web:
    print(f"Primeiro resultado: {resultados_web[0]['content'][:200]}...")

## Passo 5 — Pipeline CRAG Completo (sem LangGraph)

In [ ]:
from langchain.prompts import PromptTemplate

# Thresholds de roteamento CRAG
THRESHOLD_ALTO = 0.7   # Score >= 0.7: usar apenas docs locais
THRESHOLD_BAIXO = 0.3  # Score <  0.3: usar apenas web search
                        # 0.3 <= Score < 0.7: combinar ambos

PROMPT_GERACAO = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Você é um assistente jurídico especializado em direito brasileiro.

Use o contexto abaixo para responder à pergunta de forma precisa e objetiva.
Se o contexto não for suficiente, diga o que sabe com as informações disponíveis.

Contexto:
{context}

Pergunta: {question}

Resposta:
"""
)


def crag_pipeline(pergunta: str) -> dict:
    """
    Pipeline CRAG completo:
    1. Recuperar documentos locais
    2. Avaliar relevância (LLM-as-Judge)
    3. Rotear: local / web / híbrido
    4. Gerar resposta
    """
    print(f"\n{'='*65}")
    print(f"PERGUNTA: {pergunta}")
    print('='*65)
    
    # ─── Etapa 1: Recuperação local ───────────────────────────────
    docs = retriever.invoke(pergunta)
    print(f"\n[1] RETRIEVAL: {len(docs)} documentos recuperados")
    
    # ─── Etapa 2: Avaliação de relevância ─────────────────────────
    scores = []
    for i, doc in enumerate(docs):
        avaliacao = avaliar_relevancia(pergunta, doc.page_content)
        scores.append(avaliacao['score'])
        print(f"  Doc {i+1} score: {avaliacao['score']:.2f} — {avaliacao['justificativa'][:60]}")
    
    score_medio = sum(scores) / len(scores) if scores else 0
    print(f"  Score médio: {score_medio:.2f}")
    
    # ─── Etapa 3: Roteamento ───────────────────────────────────────
    docs_locais_texto = []
    resultados_web = []
    
    if score_medio >= THRESHOLD_ALTO:
        rota = "LOCAL"
        # Usar apenas documentos com score alto
        docs_locais_texto = [
            docs[i].page_content for i, s in enumerate(scores) if s >= THRESHOLD_ALTO
        ]
        
    elif score_medio < THRESHOLD_BAIXO:
        rota = "WEB"
        resultados_web = buscar_web(pergunta)
        
    else:  # Zona intermediária
        rota = "HIBRIDO"
        docs_locais_texto = [d.page_content for d in docs]
        resultados_web = buscar_web(pergunta)
    
    print(f"\n[2] ROTA ESCOLHIDA: {rota} (threshold alto={THRESHOLD_ALTO}, baixo={THRESHOLD_BAIXO})")
    
    # ─── Construir contexto para geração ──────────────────────────
    partes_contexto = []
    
    if docs_locais_texto:
        partes_contexto.append("=== Documentos Jurídicos Locais ===")
        for i, texto in enumerate(docs_locais_texto, 1):
            partes_contexto.append(f"[{i}] {texto}")
    
    if resultados_web:
        partes_contexto.append("\n=== Resultados de Busca Web ===")
        for i, r in enumerate(resultados_web[:2], 1):  # Limitar a 2 resultados web
            partes_contexto.append(f"[Web {i}] {r['content'][:300]}")
    
    contexto = "\n\n".join(partes_contexto)
    
    # ─── Etapa 4: Geração ─────────────────────────────────────────
    prompt_final = PROMPT_GERACAO.format(context=contexto, question=pergunta)
    resposta = llm.invoke(prompt_final).content.strip()
    
    print(f"\n[3] RESPOSTA FINAL ({rota}):")
    print(resposta)
    
    return {
        "pergunta": pergunta,
        "rota": rota,
        "score_medio": score_medio,
        "resposta": resposta,
        "docs_locais_usados": len(docs_locais_texto),
        "resultados_web_usados": len(resultados_web)
    }


print("Pipeline CRAG pronto!")

## Passo 6 — Testando as Três Rotas

In [ ]:
# Rota LOCAL: pergunta respondida pelo corpus
r1 = crag_pipeline(
    "Qual o prazo prescricional para reparação civil segundo o Código Civil?"
)

In [ ]:
# Rota WEB: pergunta sobre evento recente, não está no corpus
r2 = crag_pipeline(
    "Quais foram as principais decisões do STF sobre uso de IA no sistema judiciário em 2025?"
)

In [ ]:
# Rota HÍBRIDA: pergunta parcialmente coberta pelo corpus
r3 = crag_pipeline(
    "Como a Lei Maria da Penha tem sido aplicada em casos de violência digital recentemente?"
)

## Passo 7 — Resumo das Rotas

In [ ]:
import pandas as pd

resultados = [r1, r2, r3]
resumo = [{
    "Pergunta": r["pergunta"][:50] + "...",
    "Rota": r["rota"],
    "Score Médio": f"{r['score_medio']:.2f}",
    "Docs Locais": r["docs_locais_usados"],
    "Resultados Web": r["resultados_web_usados"]
} for r in resultados]

df = pd.DataFrame(resumo)
print("\n=== RESUMO DAS ROTAS CRAG ===")
print(df.to_string(index=False))
print("\nLimites de roteamento usados:")
print(f"  Score >= {THRESHOLD_ALTO}: rota LOCAL")
print(f"  {THRESHOLD_BAIXO} <= Score < {THRESHOLD_ALTO}: rota HÍBRIDA")
print(f"  Score < {THRESHOLD_BAIXO}: rota WEB")

## Conclusão

Neste exemplo você implementou o núcleo do CRAG sem LangGraph:

1. **Retrieval local** com ChromaDB e embeddings multilíngues
2. **Avaliador LLM-as-Judge** retornando score 0–1 com justificativa
3. **Roteamento condicional** com três caminhos: local, web e híbrido
4. **Geração contextualizada** usando a fonte adequada para cada query

No **LAB2**, você vai refatorar este pipeline usando LangGraph com `StateGraph`, tornando o fluxo mais robusto, observável e extensível com ciclos e nós paralelos.

---

*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*